# 🌍 AirSentinel Cameroun
## Notebook 01 — Chargement & Nettoyage
**Responsable : BODEHOU Sabine — ISSEA**
**Équipe : DPA Green Tech — IndabaX 2026**

### Objectif
1. Charger le dataset météo officiel
2. Charger le dataset pollution open-meteo
3. Corriger les noms de villes (accents)
4. Fusionner sur date + ville
5. Filtrer sur les 36 villes communes
6. Traiter les valeurs manquantes
7. Sauvegarder le dataset nettoyé

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../graphiques', exist_ok=True)

print('✅ Imports réussis')

✅ Imports réussis


## Étape 1 — Charger les deux datasets

In [2]:
# ── Dataset météo officiel ────────────────────────────────────────────────
df_meteo = pd.read_csv('../data/raw/dataset_meteo_cameroun.csv')

# Renommer les colonnes si nécessaire
if 'time' in df_meteo.columns:
    df_meteo = df_meteo.rename(columns={'time': 'date'})
if 'city' in df_meteo.columns:
    df_meteo = df_meteo.rename(columns={'city': 'ville'})

df_meteo['date'] = pd.to_datetime(df_meteo['date'])

print(f'Dataset météo : {df_meteo.shape[0]:,} lignes × {df_meteo.shape[1]} colonnes')
print(f'Période       : {df_meteo["date"].min().date()} → {df_meteo["date"].max().date()}')
print(f'Villes        : {df_meteo["ville"].nunique()}')
print(f'Colonnes      : {df_meteo.columns.tolist()}')

Dataset météo : 87,240 lignes × 26 colonnes
Période       : 2020-01-01 → 2025-12-20
Villes        : 40
Colonnes      : ['id', 'date', 'weather_code', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'apparent_temperature_max', 'apparent_temperature_min', 'apparent_temperature_mean', 'sunrise', 'sunset', 'daylight_duration', 'sunshine_duration', 'precipitation_sum', 'rain_sum', 'snowfall_sum', 'precipitation_hours', 'wind_speed_10m_max', 'wind_gusts_10m_max', 'wind_direction_10m_dominant', 'shortwave_radiation_sum', 'et0_fao_evapotranspiration', 'ville', 'region', 'latitude', 'longitude']


In [3]:
# ── Dataset pollution open-meteo ─────────────────────────────────────────
df_poll = pd.read_excel('../data/raw/donnees_qualite_air_FINAL.xlsx',
                         sheet_name='Données complètes')
df_poll['date'] = pd.to_datetime(df_poll['date'])

print(f'Dataset pollution : {df_poll.shape[0]:,} lignes × {df_poll.shape[1]} colonnes')
print(f'Période           : {df_poll["date"].min().date()} → {df_poll["date"].max().date()}')
print(f'Villes            : {df_poll["ville"].nunique()}')

Dataset pollution : 87,680 lignes × 29 colonnes
Période           : 2020-01-01 → 2025-12-31
Villes            : 40


## Étape 2 — Vérification

In [4]:

# Justification : harmonisation nécessaire avant fusion

# Vérifier les villes communes
villes_meteo = set(df_meteo['ville'].unique())
villes_poll  = set(df_poll['ville'].unique())
villes_communes = villes_meteo & villes_poll

print(f'\nVilles communes : {len(villes_communes)}')
print(f'Villes météo seules : {villes_meteo - villes_poll}')
print(f'Villes pollution seules : {villes_poll - villes_meteo}')


Villes communes : 40
Villes météo seules : set()
Villes pollution seules : set()


## Étape 3 — Fusionner les deux datasets

In [5]:
# Colonnes à garder depuis le dataset pollution
cols_poll = ['date', 'ville', 'pm2_5_moyen', 'pm2_5_max', 'pm10_moyen',
             'dust_moyen', 'co_moyen', 'no2_moyen', 'so2_moyen',
             'ozone_moyen', 'uv_moyen', 'us_aqi_moyen',
             'polluant_dominant', 'niveau_alerte']

# Garder seulement les colonnes disponibles
cols_poll = [c for c in cols_poll if c in df_poll.columns]

# Filtrer sur les 36 villes communes et période 2022-2025
df_meteo_f = df_meteo[
    (df_meteo['ville'].isin(villes_communes)) &
    (df_meteo['date'] >= '2022-08-05')
].copy()

df_poll_f = df_poll[df_poll['ville'].isin(villes_communes)].copy()

# Fusion
df = pd.merge(df_meteo_f, df_poll_f[cols_poll],
              on=['date', 'ville'], how='left')

print(f'Dataset fusionné : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'Villes           : {df["ville"].nunique()}')
print(f'Période          : {df["date"].min().date()} → {df["date"].max().date()}')

Dataset fusionné : 49,360 lignes × 38 colonnes
Villes           : 40
Période          : 2022-08-05 → 2025-12-20


## Étape 4 — Traiter les valeurs manquantes

In [6]:
# Vérifier les données manquantes
print('Données manquantes par variable :')
manquants = df.isnull().sum()
manquants_pct = (df.isnull().mean() * 100).round(1)
for col in manquants[manquants > 0].index:
    print(f'  {col:30s} : {manquants[col]:,} ({manquants_pct[col]}%)')

Données manquantes par variable :


In [7]:
# Imputation par médiane ville + mois
# Méthode : médiane par groupe (standard statistique)
df['mois'] = df['date'].dt.month

vars_a_imputer = ['pm2_5_moyen', 'pm10_moyen', 'dust_moyen',
                  'co_moyen', 'no2_moyen', 'so2_moyen',
                  'ozone_moyen', 'uv_moyen']

for col in vars_a_imputer:
    if col in df.columns:
        avant = df[col].isna().sum()
        df[col] = df.groupby(['ville', 'mois'])[col].transform(
            lambda x: x.fillna(x.median())
        )
        apres = df[col].isna().sum()
        if avant > 0:
            print(f'  {col:25s} : {avant:,} → {apres:,} manquants')

df.drop('mois', axis=1, inplace=True)
print('✅ Valeurs manquantes traitées')

✅ Valeurs manquantes traitées


## Étape 5 — Statistiques descriptives

In [8]:
print('=' * 60)
print('STATISTIQUES DESCRIPTIVES')
print('=' * 60)
print(f'Lignes totales       : {len(df):,}')
print(f'Villes               : {df["ville"].nunique()}')
print(f'Régions              : {df["region"].nunique()}')
print(f'Période              : {df["date"].min().date()} → {df["date"].max().date()}')
print()
print(f'PM2.5 moyen national : {df["pm2_5_moyen"].mean():.2f} µg/m³')
print(f'Norme OMS            : 15 µg/m³')
print(f'Dépassement OMS      : {(df["pm2_5_moyen"] > 15).mean()*100:.1f}% des jours')

v_max = df.groupby('ville')['pm2_5_moyen'].mean()
print(f'\nVille + polluée      : {v_max.idxmax()} ({v_max.max():.2f} µg/m³)')
print(f'Ville + saine        : {v_max.idxmin()} ({v_max.min():.2f} µg/m³)')
print('=' * 60)

STATISTIQUES DESCRIPTIVES
Lignes totales       : 49,360
Villes               : 40
Régions              : 10
Période              : 2022-08-05 → 2025-12-20

PM2.5 moyen national : 20.18 µg/m³
Norme OMS            : 15 µg/m³
Dépassement OMS      : 51.5% des jours

Ville + polluée      : Mokolo (30.81 µg/m³)
Ville + saine        : Kribi (10.05 µg/m³)


## Étape 6 — Sauvegarder

In [9]:
df.to_excel('../data/processed/dataset_nettoye.xlsx', index=False)
print('✅ Dataset nettoyé sauvegardé : data/processed/dataset_nettoye.xlsx')
print(f'   {len(df):,} lignes × {df.shape[1]} colonnes')

✅ Dataset nettoyé sauvegardé : data/processed/dataset_nettoye.xlsx
   49,360 lignes × 38 colonnes
